# AI Bubble Detection — Visualization

This notebook only reads results from disk. To refresh results, run the
scripts from the project root first:

```bash
python scripts/01_download_data.py
python scripts/02_run_montecarlo.py      # slow — run once
python scripts/03_run_analysis.py --methods all
```

Then re-run this notebook. Modifying anything here (plots, tables) does
not require re-running the analysis.

**Erika Francesca Caragnano — Politecnico di Torino**

In [1]:
!pip install pandas matplotlib pyarrow fastparquet statsmodels yfinance numba

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Force Python to forget all cached src modules
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110})

from config import settings
from src import api


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## ① Cross-method comparison table

In [4]:
comparison = api.get_comparison_table()
print(f"{len(comparison)} tickers")
comparison

55 tickers


,Ticker,Segment,T,ADF reject,PP reject,KPSS reject,SADF stat,SADF signal,GSADF stat,GSADF cv,GSADF reject,# Episodes,SV-ADF window,SV-ADF episode,SV-ADF collapse
0,ASML,1. Semiconductor Equipment,1842,·,·,✓,1.67,✓,2.00,2.39,·,1,2022-11-01_2026-05-01,·,NaN
1,AMAT,1. Semiconductor Equipment,1842,·,·,✓,1.80,✓,2.22,2.39,·,2,2022-11-01_2026-05-01,·,NaN
2,LRCX,1. Semiconductor Equipment,1842,·,·,✓,4.49,✓,4.52,2.39,✓,3,2022-11-01_2026-05-01,✓,formal
3,NVDA,2. Chip Design (AI accelerators),1842,·,·,✓,6.34,✓,6.34,2.39,✓,4,2022-11-01_2026-05-01,✓,formal
4,AMD,2. Chip Design (AI accelerators),1842,·,·,✓,1.34,·,2.90,2.39,✓,1,2022-11-01_2026-05-01,·,NaN
5,INTC,2. Chip Design (AI accelerators),1842,·,·,✓,0.32,·,3.89,2.39,✓,0,2022-11-01_2026-05-01,·,NaN
6,AVGO,2. Chip Design (AI accelerators),1842,·,·,✓,4.00,✓,4.00,2.39,✓,3,2022-11-01_2026-05-01,·,NaN
7,MRVL,2. Chip Design (AI accelerators),1842,·,·,✓,1.80,✓,3.60,2.39,✓,0,2022-11-01_2026-05-01,·,NaN
8,MU,3. Memory (HBM-relevant),1842,·,·,✓,7.83,✓,7.86,2.39,✓,3,2022-11-01_2026-05-01,✓,end_of_sample
9,TSM,4. Foundry (leading-edge),1842,·,·,✓,2.92,✓,2.92,2.39,✓,5,2022-11-01_2026-05-01,·,NaN


## ② Segment-level summary

In [5]:
api.get_segment_summary()

,Segment,N,Tickers with GSADF episode,Total episodes
0,0. Index / Control,3,0,0
1,1. Semiconductor Equipment,3,3,6
2,10. Application & AI Platform Software,4,2,2
3,11. Edge AI & Devices,3,3,4
4,2. Chip Design (AI accelerators),5,3,8
5,3. Memory (HBM-relevant),1,1,3
6,4. Foundry (leading-edge),1,1,5
7,5. Storage (non-HBM control),2,2,4
8,6. Data-Centre Networking & Optical,4,3,4
9,7. Data-Centre REITs (colocation & build-out),2,0,0


## ③ Master episode list

In [6]:
episodes = api.get_episodes_table()
print(f"{len(episodes)} episodes")
episodes

96 episodes


,Ticker,Segment,Method,Window,Start,End,Duration (days),Collapse type
0,GLD,C. Confound: Precious metals (monetary),GSADF,2019-01-02 → 2026-04-30,2019-06-20,2019-09-24,96,NaN
1,TSM,4. Foundry (leading-edge),GSADF,2019-01-02 → 2026-04-30,2019-10-15,2020-01-17,94,NaN
2,LRCX,1. Semiconductor Equipment,GSADF,2019-01-02 → 2026-04-30,2019-10-25,2020-02-19,117,NaN
3,TSLA,11. Edge AI & Devices,GSADF,2019-01-02 → 2026-04-30,2019-10-25,2020-03-05,132,NaN
4,AAPL,11. Edge AI & Devices,GSADF,2019-01-02 → 2026-04-30,2019-10-28,2020-02-21,116,NaN
...,...,...,...,...,...,...,...,...
91,LRCX,1. Semiconductor Equipment,SV-ADF,2022-11-01_2026-05-01,2025-12-18,2026-03-02,74,formal
92,AMAT,1. Semiconductor Equipment,GSADF,2019-01-02 → 2026-04-30,2026-01-06,2026-04-30,114,NaN
93,DBC,C. Benchmark: Broad commodity indices,GSADF,2019-01-02 → 2026-04-30,2026-01-27,2026-04-30,93,NaN
94,PDBC,C. Benchmark: Broad commodity indices,GSADF,2019-01-02 → 2026-04-30,2026-01-27,2026-04-30,93,NaN


## ④ Per-ticker GSADF vs SV-ADF comparison plots

Both methods on a common timeline. GSADF uses the fixed window
(2020-01-01 → 2026-05-01). **Both SV-ADF windows are shown on the same
panel**: W1 post-ChatGPT (orange, 2021-11-01 → 2026-05-01) and W2
pre-ChatGPT (teal, 2020-01-01 → 2022-10-31).

In [ ]:
for tk in settings.ALL_TICKERS:
    methods = api.list_methods_for(tk)
    if "gsadf" not in methods or "svadf" not in methods:
        continue
    api.get_comparison_figure(tk)
    plt.show()

## ⑤ GSADF-only diagnostic plots

In [ ]:
for tk in settings.ALL_TICKERS:
    if "gsadf" not in api.list_methods_for(tk):
        continue
    api.get_gsadf_figure(tk)
    plt.show()

## ⑥ GSADF segment panels

One panel per segment — all tickers in the same category laid out side by side,
sharing a common timeline. Red shading marks explosive episodes detected by GSADF.

> **Saving:** run `python scripts/05_generate_figures.py --types gsadf_panel` to save all panels to `outputs/figures/gsadf_panels/`.

In [ ]:
# Reload src modules so any edits to plotting code are picked up without a kernel restart
for _key in list(sys.modules.keys()):
    if _key.startswith("src"):
        del sys.modules[_key]

from src.plotting.diagnostic import plot_gsadf_segment_panel

all_segments = list(dict.fromkeys(
    seg for _, (seg, _) in settings.SEGMENT_OF.items()
))

for seg in all_segments:
    plot_gsadf_segment_panel(seg)  # displays inline — batch saving is via 05_generate_figures.py

## ⑦ Export tables

In [ ]:
settings.TABLES_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(settings.TABLES_SUMMARY_DIR / "comparison_table.csv", index=False)
episodes.to_csv (settings.TABLES_SUMMARY_DIR / "episodes_table.csv",  index=False)
api.get_segment_summary().to_csv(settings.TABLES_SUMMARY_DIR / "segment_summary.csv", index=False)
print(f"Saved → {settings.TABLES_SUMMARY_DIR}")

In [11]:
from src.io.results import available_methods
print(available_methods("IBM"))

['stationarity', 'sadf', 'gsadf', 'svadf']
